# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test_name per calendar month) from silver lab_results, with calendar month/quarter names and a LAG-based month-over-month abnormal rate trend, per the Silver-to-Gold STTM. Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT.

Note: `silver.calendar` has no `quarter_name` column, so `quarter_name` is derived as `'Q' || calendar.quarter` (e.g. "Q1").

In [0]:
dbutils.widgets.text("catalog", "abd_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

silver = f"{catalog}.silver"
gold = f"{catalog}.gold"
target_table = f"{gold}.lab_test_trends"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH monthly AS (
    SELECT
        test_name,
        YEAR(test_date)                            AS year,
        MONTH(test_date)                           AS month,
        DATE_TRUNC('MONTH', test_date)             AS month_start,
        COUNT(*)                                   AS total_tests,
        COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')       AS critical_test_count
    FROM {silver}.lab_results
    GROUP BY test_name, YEAR(test_date), MONTH(test_date), DATE_TRUNC('MONTH', test_date)
),
rates AS (
    SELECT
        m.*,
        CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
        CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
    FROM monthly m
),
with_lag AS (
    SELECT
        r.*,
        LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
    FROM rates r
)
SELECT
    CAST(w.test_name AS STRING)                                         AS test_name,
    CAST(w.year AS INT)                                                 AS year,
    CAST(w.month AS INT)                                                AS month,
    CAST(c.month_name AS STRING)                                        AS month_name,
    CAST(CONCAT('Q', c.quarter) AS STRING)                              AS quarter_name,
    CAST(w.total_tests AS INT)                                          AS total_tests,
    CAST(w.abnormal_test_count AS INT)                                  AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                                  AS critical_test_count,
    w.abnormal_rate_pct,
    w.critical_rate_pct,
    CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                AS prev_month_abnormal_rate_pct,
    CASE
        WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
        ELSE CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
    END                                                                 AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {silver}.calendar c
    ON c.date_key = CAST(date_format(w.month_start, 'yyyyMMdd') AS INT)
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))